# Feature Engineering — Per-Department Comment Features

Adds 50 new columns to `master_dataset.csv` using plan comments data:

- **ReviewCycleCount_[Dept]** — how many times each department sent the permit back for corrections (delay signal)
- **CommentCount_[Dept]** — how many issues each department flagged in total (volume signal)

Output saved as `master_dataset_comments_features.csv`.

In [ ]:
import pandas as pd
import os

# Update these paths to match your machine
DATA_DIR = '/Users/mk/Desktop/CSB425-City-of-Seattle-Permit-Predictor/data'

COMMENTS_PATH = os.path.join(DATA_DIR, 'plan_comments.csv')
MASTER_PATH   = os.path.join(DATA_DIR, 'master_dataset.csv')
OUTPUT_PATH   = os.path.join(DATA_DIR, 'master_dataset_comments_features.csv')

In [17]:
# Load both files
df_comments = pd.read_csv(COMMENTS_PATH)
master      = pd.read_csv(MASTER_PATH)

# Keep only comments from 2021 onwards — data quality drops before that (per Dev)
df_comments['DocumentDate'] = pd.to_datetime(df_comments['DocumentDate'], errors='coerce')
df_comments = df_comments[df_comments['DocumentDate'].dt.year >= 2021].copy()

# Remove rows where ReviewType is an author name instead of a department
df_comments = df_comments[~df_comments['ReviewType'].str.startswith('Author:', na=False)]

# Remove junk rows from parsing errors
df_comments = df_comments[~df_comments['ReviewType'].str.contains('Page Index:', na=False)]

# Collapse the two Ordinance Structural variations into one
df_comments['ReviewType'] = df_comments['ReviewType'].replace({
    'Ordinance/Structural': 'Ordinance_Structural',
    'Ordinance Structural': 'Ordinance_Structural'
})

print(f'Comments rows : {len(df_comments):,}')
print(f'Master rows   : {len(master):,}')

Comments rows : 26,293
Master rows   : 14,201


In [ ]:
# Feature 1: ReviewCycleCount per department
# For each permit, count how many distinct review cycles each department left comments in.
# Ex: ReviewCycleCount_Zoning = 3 means Zoning pushed back 3 separate times.

cycle_counts = (
    df_comments
    .groupby(['PermitNum', 'ReviewType'])['ReviewCycle']
    .nunique()
    .unstack(fill_value=0)
)

# Rename columns: e.g. 'Zoning' -> 'ReviewCycleCount_Zoning'
cycle_counts.columns = [
    'ReviewCycleCount_' + col.replace(' ', '_').replace('/', '_')
    for col in cycle_counts.columns
]

print(f'Columns created: {cycle_counts.columns.tolist()}')

Columns created: ['ReviewCycleCount_Addressing', 'ReviewCycleCount_City_Light', 'ReviewCycleCount_Conveyance', 'ReviewCycleCount_Design_Review', 'ReviewCycleCount_Drainage', 'ReviewCycleCount_ECA_GeoTech', 'ReviewCycleCount_ECA_Riparian', 'ReviewCycleCount_ECA_Wetland', 'ReviewCycleCount_ECA_Wildlife', 'ReviewCycleCount_Energy', 'ReviewCycleCount_Fire', 'ReviewCycleCount_Floodplain', 'ReviewCycleCount_Geo_Soils', 'ReviewCycleCount_Housing', 'ReviewCycleCount_Land_Use', 'ReviewCycleCount_MHA', 'ReviewCycleCount_Mechanical', 'ReviewCycleCount_Ordinance', 'ReviewCycleCount_Ordinance_Structural', 'ReviewCycleCount_Policy', 'ReviewCycleCount_Revegetation', 'ReviewCycleCount_Shoreline', 'ReviewCycleCount_Structural_Engineer', 'ReviewCycleCount_Tree', 'ReviewCycleCount_Zoning']


In [ ]:
# Feature 2: CommentCount per department
# For each permit, count the total number of comments each department left.
# Ex: CommentCount_Drainage = 12 means Drainage flagged 12 individual issues.

comment_counts = (
    df_comments
    .groupby(['PermitNum', 'ReviewType'])
    .size()
    .unstack(fill_value=0)
)

# Rename columns: ex: 'Zoning' -> 'CommentCount_Zoning'
comment_counts.columns = [
    'CommentCount_' + col.replace(' ', '_').replace('/', '_')
    for col in comment_counts.columns
]

print(f'Columns created: {comment_counts.columns.tolist()}')

Columns created: ['CommentCount_Addressing', 'CommentCount_City_Light', 'CommentCount_Conveyance', 'CommentCount_Design_Review', 'CommentCount_Drainage', 'CommentCount_ECA_GeoTech', 'CommentCount_ECA_Riparian', 'CommentCount_ECA_Wetland', 'CommentCount_ECA_Wildlife', 'CommentCount_Energy', 'CommentCount_Fire', 'CommentCount_Floodplain', 'CommentCount_Geo_Soils', 'CommentCount_Housing', 'CommentCount_Land_Use', 'CommentCount_MHA', 'CommentCount_Mechanical', 'CommentCount_Ordinance', 'CommentCount_Ordinance_Structural', 'CommentCount_Policy', 'CommentCount_Revegetation', 'CommentCount_Shoreline', 'CommentCount_Structural_Engineer', 'CommentCount_Tree', 'CommentCount_Zoning']


In [ ]:
# Merge new features onto master dataset
# Normalize permit numbers to uppercase to avoid join mismatches

master_v2 = master.copy()
master_v2['_key'] = master_v2['permitnum'].str.upper().str.strip()
cycle_counts.index   = cycle_counts.index.str.upper().str.strip()
comment_counts.index = comment_counts.index.str.upper().str.strip()

master_v2 = master_v2.merge(cycle_counts,   left_on='_key', right_index=True, how='left')
master_v2 = master_v2.merge(comment_counts, left_on='_key', right_index=True, how='left')
master_v2 = master_v2.drop(columns=['_key'])

# Permits with no comments get 0 instead of NaN
new_cols = [c for c in master_v2.columns if c.startswith('ReviewCycleCount_') or c.startswith('CommentCount_')]
for c in new_cols:
    master_v2[c] = master_v2[c].fillna(0).astype(int)

print(f'New shape: {master_v2.shape[0]:,} rows x {master_v2.shape[1]} columns')
print(f'New columns added: {len(new_cols)}')

New shape: 14,201 rows x 95 columns
New columns added: 50


In [ ]:
# Sanity check
# Look at the 5 permits with the highest Zoning cycle count
# and verify the numbers match the raw comments data

top5 = master_v2.nlargest(5, 'ReviewCycleCount_Zoning')[[
    'permitnum', 'ReviewCycleCount_Zoning', 'CommentCount_Zoning', 'totaldaysplanreview'
]]
print('Top 5 permits by Zoning cycle count:')
print(top5.to_string())

# Double-check the top permit against the raw data
top_permit = top5.iloc[0]['permitnum']
raw = df_comments[
    (df_comments['PermitNum'].str.upper() == top_permit.upper()) &
    (df_comments['ReviewType'] == 'Zoning')
]
print(f'\nVerification for {top_permit} — Zoning:')
print(f'  Distinct cycles in raw data : {raw["ReviewCycle"].nunique()}')
print(f'  Total comments in raw data  : {len(raw)}')

Top 5 permits by Zoning cycle count:
       permitnum  ReviewCycleCount_Zoning  CommentCount_Zoning  totaldaysplanreview
7297  6932686-CN                        6                    6                594.0
8948  6994052-CN                        6                   12                383.0
1039  6824832-CN                        5                   29               1069.0
2860  6878392-CN                        5                    6                442.0
6155  6989637-CN                        5                   24                379.0

Verification for 6932686-CN — Zoning:
  Distinct cycles in raw data : 6
  Total comments in raw data  : 6


In [ ]:
# Save
master_v2.to_csv(OUTPUT_PATH, index=False)
print(f'Saved: master_dataset_comments_features.csv')
print(f'  {master_v2.shape[0]:,} rows, {master_v2.shape[1]} columns')

Saved: master_dataset_comments_features.csv
  14,201 rows, 95 columns
